In [1]:
import requests
import re
import json

url = "http://182.72.188.195/opac-tmpl/bootstrap/images/OADATABASE/OA%20Databases.html"

response = requests.get(url, timeout=30)
response.raise_for_status()

print("Page downloaded:", len(response.text), "characters")
print("Status:", response.status_code)

Page downloaded: 28869 characters
Status: 200


In [4]:
import subprocess
import sys

try:
    import chompjs
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "chompjs"])
    import chompjs

match = re.search(
    r'const\s+RESOURCES\s*=\s*(\[.*?\])\s*;',
    response.text,
    re.DOTALL
)

if match:
    try:
        # chompjs.parse_js_object can parse standard JS/JSON objects with unquoted keys, single quotes, etc.
        resources = chompjs.parse_js_object(match.group(1))
        print("Records extracted:", len(resources))
        print("First record:")
        print(json.dumps(resources[0], indent=2))
    except Exception as e:
        print("Error parsing array using chompjs:", e)
        print("Raw JS string starts with:", match.group(1)[:500])
else:
    print("RESOURCES array not found.")

Records extracted: 20
First record:
{
  "title": "Indian Kanoon",
  "url": "https://indiankanoon.org/",
  "category": "Law & Judiciary",
  "desc": "Free repository of Indian case law, statutes, and legal documents",
  "icon": "i-scale"
}


In [5]:
import pandas as pd
import re

# Convert extracted records into a DataFrame
df = pd.DataFrame(resources)

# Rename desc to description
df.rename(columns={"desc": "description"}, inplace=True)

# Clean Markdown formatting from URLs
df["url"] = df["url"].apply(
    lambda x: re.sub(r'\[([^\]]+)\]\((https?://[^)]+)\)', r'\2', x)
)

# Display all records
display(df)

# Check dataset structure
print("Dataset shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nMissing values:")
print(df.isnull().sum())

,title,url,category,description,icon
0,Indian Kanoon,https://indiankanoon.org/,Law & Judiciary,"Free repository of Indian case law, statutes, ...",i-scale
1,eCourts NJDG,https://njdg.ecourts.gov.in/njdg_v3/,Law & Judiciary,National Judicial Data Grid: real-time pendenc...,i-gavel
2,WIPO,https://www.wipo.int/portal/en/,Law & Judiciary,World Intellectual Property Organization globa...,i-globe
3,RePEc,https://ideas.repec.org/,Economics & Finance,Research Papers in Economics: largest open eco...,i-bulb
4,RBI,https://www.rbi.org.in/home.aspx,Economics & Finance,"Reserve Bank of India publications, data, and ...",i-bank
5,India Data Portal,https://indiadataportal.com/,Statistics & Data,Curated open datasets on the Indian economy an...,i-database
6,IMF eLibrary,https://www.elibrary.imf.org/,Economics & Finance,International Monetary Fund publications and w...,i-book
7,Tijori Finance,https://www.tijorifinance.com/?r=n,Economics & Finance,Indian stock and company financial data explorer,i-line
8,MOSPI,https://www.mospi.gov.in/,Statistics & Data,Ministry of Statistics & Programme Implementat...,i-bar
9,Data.gov.in,https://www.data.gov.in/,Statistics & Data,Open Government Data Platform of India,i-bar


Dataset shape: (20, 5)

Columns: ['title', 'url', 'category', 'description', 'icon']

Missing values:
title          0
url            0
category       0
description    0
icon           0
dtype: int64


In [6]:
df.to_csv("oa_databases.csv", index=False, encoding="utf-8-sig")
df.to_excel("oa_databases.xlsx", index=False)

print("CSV and Excel datasets created successfully!")

CSV and Excel datasets created successfully!


In [7]:
# Check whether URLs are plain URLs
for i, url in enumerate(df["url"]):
    print(i, url)

# Check duplicate records
print("\nDuplicate titles:", df["title"].duplicated().sum())

# Check empty strings
print("\nEmpty strings:")
print((df == "").sum())

# Check data types
print("\nData types:")
print(df.dtypes)

0 https://indiankanoon.org/
1 https://njdg.ecourts.gov.in/njdg_v3/
2 https://www.wipo.int/portal/en/
3 https://ideas.repec.org/
4 https://www.rbi.org.in/home.aspx
5 https://indiadataportal.com/
6 https://www.elibrary.imf.org/
7 https://www.tijorifinance.com/?r=n
8 https://www.mospi.gov.in/
9 https://www.data.gov.in/
10 https://core.ac.uk/
11 https://nap.nationalacademies.org/
12 https://www.semanticscholar.org/
13 https://doaj.org/
14 https://www.doabooks.org/
15 https://www.techrxiv.org/
16 https://www.springeropen.com/
17 https://southasiacommons.net/
18 https://shodhganga.inflibnet.ac.in/
19 https://sites.google.com/view/puloar/home

Duplicate titles: 0

Empty strings:
title          0
url            0
category       0
description    0
icon           0
dtype: int64

Data types:
title          object
url            object
category       object
description    object
icon           object
dtype: object


In [8]:
df["url"] = df["url"].str.extract(r'\((https?://[^)]+)\)', expand=False).fillna(df["url"])

In [9]:
df.to_csv("oa_databases.csv", index=False, encoding="utf-8-sig")
df.to_excel("oa_databases.xlsx", index=False)

print("Saved successfully!")

Saved successfully!


In [10]:
from google.colab import files

files.download("oa_databases.csv")
files.download("oa_databases.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
!pip install -q sentence-transformers faiss-cpu

In [12]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


In [13]:
# Combine the useful fields into one searchable text per database

df["content"] = (
    "Database: " + df["title"].astype(str) +
    "\nCategory: " + df["category"].astype(str) +
    "\nDescription: " + df["description"].astype(str) +
    "\nWebsite: " + df["url"].astype(str)
)

# Inspect the first record
print(df["content"].iloc[0])

Database: Indian Kanoon
Category: Law & Judiciary
Description: Free repository of Indian case law, statutes, and legal documents
Website: https://indiankanoon.org/


In [14]:
# Convert all 20 database descriptions into embeddings

texts = df["content"].tolist()

embeddings = model.encode(
    texts,
    convert_to_numpy=True,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Embedding shape:", embeddings.shape)
print("Number of records:", len(embeddings))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding shape: (20, 384)
Number of records: 20


In [16]:
import faiss
import numpy as np

# Create a FAISS index
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

# Add all embeddings
index.add(embeddings.astype("float32"))

print("FAISS index created!")
print("Total vectors stored:", index.ntotal)

FAISS index created!
Total vectors stored: 20


In [17]:
def search_database(query, top_k=5):

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(query_embedding, top_k)

    for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), 1):

        print(f"\nRank: {rank}")
        print("Database:", df.iloc[idx]["title"])
        print("Category:", df.iloc[idx]["category"])
        print("Description:", df.iloc[idx]["description"])
        print("Website:", df.iloc[idx]["url"])
        print("Similarity score:", round(float(score), 4))

In [18]:
search_database("Where can I find Indian legal judgments and case laws?")


Rank: 1
Database: Indian Kanoon
Category: Law & Judiciary
Description: Free repository of Indian case law, statutes, and legal documents
Website: https://indiankanoon.org/
Similarity score: 0.7168

Rank: 2
Database: WIPO
Category: Law & Judiciary
Description: World Intellectual Property Organization global IP portal
Website: https://www.wipo.int/portal/en/
Similarity score: 0.4303

Rank: 3
Database: Data.gov.in
Category: Statistics & Data
Description: Open Government Data Platform of India
Website: https://www.data.gov.in/
Similarity score: 0.3781

Rank: 4
Database: DOAJ
Category: Journals & Books
Description: Directory of Open Access Journals
Website: https://doaj.org/
Similarity score: 0.3622

Rank: 5
Database: eCourts NJDG
Category: Law & Judiciary
Description: National Judicial Data Grid: real-time pendency and disposal stats
Website: https://njdg.ecourts.gov.in/njdg_v3/
Similarity score: 0.36


In [19]:
search_database("I need economic research papers and financial data.")


Rank: 1
Database: RePEc
Category: Economics & Finance
Description: Research Papers in Economics: largest open economics bibliography
Website: https://ideas.repec.org/
Similarity score: 0.6061

Rank: 2
Database: IMF eLibrary
Category: Economics & Finance
Description: International Monetary Fund publications and working papers
Website: https://www.elibrary.imf.org/
Similarity score: 0.491

Rank: 3
Database: RBI
Category: Economics & Finance
Description: Reserve Bank of India publications, data, and reports
Website: https://www.rbi.org.in/home.aspx
Similarity score: 0.4303

Rank: 4
Database: India Data Portal
Category: Statistics & Data
Description: Curated open datasets on the Indian economy and society
Website: https://indiadataportal.com/
Similarity score: 0.4184

Rank: 5
Database: CORE
Category: Academic Research
Description: World's largest aggregator of open access research papers
Website: https://core.ac.uk/
Similarity score: 0.4028


In [20]:
search_database("Where can I find open access research journals?")


Rank: 1
Database: DOAJ
Category: Journals & Books
Description: Directory of Open Access Journals
Website: https://doaj.org/
Similarity score: 0.7856

Rank: 2
Database: SpringerOpen
Category: Journals & Books
Description: Springer's portfolio of fully open access journals
Website: https://www.springeropen.com/
Similarity score: 0.7131

Rank: 3
Database: CORE
Category: Academic Research
Description: World's largest aggregator of open access research papers
Website: https://core.ac.uk/
Similarity score: 0.7038

Rank: 4
Database: DOAB
Category: Journals & Books
Description: Directory of Open Access Books
Website: https://www.doabooks.org/
Similarity score: 0.6989

Rank: 5
Database: NAP
Category: Journals & Books
Description: National Academies Press: free PDFs of academic publications
Website: https://nap.nationalacademies.org/
Similarity score: 0.5678


In [21]:
search_database("I want Indian government statistics and datasets.")


Rank: 1
Database: Data.gov.in
Category: Statistics & Data
Description: Open Government Data Platform of India
Website: https://www.data.gov.in/
Similarity score: 0.8073

Rank: 2
Database: MOSPI
Category: Statistics & Data
Description: Ministry of Statistics & Programme Implementation, Government of India
Website: https://www.mospi.gov.in/
Similarity score: 0.7013

Rank: 3
Database: India Data Portal
Category: Statistics & Data
Description: Curated open datasets on the Indian economy and society
Website: https://indiadataportal.com/
Similarity score: 0.67

Rank: 4
Database: Indian Kanoon
Category: Law & Judiciary
Description: Free repository of Indian case law, statutes, and legal documents
Website: https://indiankanoon.org/
Similarity score: 0.4964

Rank: 5
Database: RBI
Category: Economics & Finance
Description: Reserve Bank of India publications, data, and reports
Website: https://www.rbi.org.in/home.aspx
Similarity score: 0.4794


In [22]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ), "GB")

CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [23]:
!pip install -q transformers accelerate bitsandbytes

In [24]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "Qwen/Qwen2.5-3B-Instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

llm = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)

print("Qwen model loaded successfully!")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen model loaded successfully!


In [25]:
def rag_chat(question, top_k=5):

    # 1. Convert question into embedding
    query_embedding = model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    # 2. Retrieve relevant records from FAISS
    scores, indices = index.search(query_embedding, top_k)

    retrieved_context = []

    for idx in indices[0]:
        if idx == -1:
            continue

        record = df.iloc[idx]

        retrieved_context.append(
            f"""
            Database: {record['title']}
            Category: {record['category']}
            Description: {record['description']}
            Website: {record['url']}
            """
        )

    context = "\n".join(retrieved_context)

    # 3. Give retrieved information to Qwen
    messages = [
        {
            "role": "system",
            "content": """
            You are a helpful Presidency University Open Access
            Database Assistant.

            Answer questions using ONLY the provided context.

            Rules:
            1. Do not invent database names or URLs.
            2. If the answer is not in the context, say:
               "I could not find this information in the available records."
            3. Explain which database is useful and why.
            4. Include the official website when available.
            5. Be clear, accurate and beginner-friendly.
            """
        },
        {
            "role": "user",
            "content": f"""
            Available information:

            {context}

            User question: {question}

            Answer based only on the available information.
            """
        }
    ]

    # 4. Generate response
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(llm.device)

    with torch.no_grad():
        outputs = llm.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return response

In [26]:
question = "Where can I find Indian legal judgments and case laws?"

answer = rag_chat(question)

print("You:", question)
print("\nChatbot:", answer)

You: Where can I find Indian legal judgments and case laws?

Chatbot: You can find Indian legal judgments and case laws on the Indian Kanoon database. This database is a free repository of Indian case law, statutes, and legal documents. You can access it at [https://indiankanoon.org/](https://indiankanoon.org/).

Database Useful: Indian Kanoon
Reason: This database specifically mentions that it is a "Free repository of Indian case law, statutes, and legal documents," making it the most relevant for finding Indian legal judgments and case laws among the given options.


In [28]:

print(rag_chat("I need Indian economic and financial information."))

Based on the available information, you would be best served by using the Tijori Finance database. This database provides Indian stock and company financial data, which falls under the category of Economics & Finance. It's specifically designed to explore Indian stock and company financial data, making it highly relevant for your needs.

### Useful Database:
**Tijori Finance**

### Why This Is Useful:
- **Category**: Economics & Finance
- **Description**: Provides Indian stock and company financial data
- **Website**: [https://www.tijorifinance.com/?r=n](https://www.tijorifinance.com/?r=n)

This database directly addresses your request for Indian economic and financial information.


In [33]:
from google.colab import userdata

try:
    # Always pass the string name of the secret key in quotes
    API_TOKEN = userdata.get("INDIANKANOON_API_TOKEN")
    print("API token loaded successfully!")
except userdata.SecretNotFoundError:
    API_TOKEN = None
    print("Warning: 'INDIANKANOON_API_TOKEN' secret key not found in Colab Secrets.")
    print("To fix this, click the Key (Secrets) icon on the left panel, add a key named 'INDIANKANOON_API_TOKEN', and paste your token value.")

print("API token loaded:", bool(API_TOKEN))

To fix this, click the Key (Secrets) icon on the left panel, add a key named 'INDIANKANOON_API_TOKEN', and paste your token value.
API token loaded: False


In [34]:
!pip install -q gradio

In [35]:
import gradio as gr

def chatbot_response(message, history):

    if not message.strip():
        return "Please enter a question."

    try:
        answer = rag_chat(message, top_k=5)
        return answer

    except Exception as e:
        return f"An error occurred: {str(e)}"


demo = gr.ChatInterface(
    fn=chatbot_response,
    title="🎓 University Research Assistant",
    description=(
        "Ask questions about open-access databases, "
        "academic resources, legal information and research."
    ),
    examples=[
        "Where can I find Indian legal judgments?",
        "Suggest databases for economic research.",
        "Where can I find open-access journals?",
        "I need Indian government statistics.",
        "Where can I find PhD theses?",
        "Which databases provide engineering research?"
    ],
    chatbot=gr.Chatbot(height=500)
)

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://62a36f37b22489c926.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://62a36f37b22489c926.gradio.live
